# 06. Финальная проверка за февраль

После подбора из 05 выбираем pipeline с лучшим январским NDCG@10: сохранённый CatBoost и соответствующие ему настройки ALS/KNN. CatBoost обучен на снимках сентября–декабря; перед тестом не переобучается. Retrieval использует историю до 1 февраля. Полный февральский ground truth применяется только для оценки.

Февраль уже был просмотрен до дополнительного подбора параметров. Перестановка ноутбуков исправляет порядок воспроизведения, но не делает повторную проверку независимой.

In [1]:
from pathlib import Path
import pandas as pd
from catboost import CatBoostRanker

from movielens_recsys.data import load_movielens
from movielens_recsys.ranking import build_snapshot, split_snapshot, select_best_pipeline
from movielens_recsys.evaluation import evaluate_recommendations
if (Path.cwd() / "../movielens-reccomender/configs/experiment.json").exists():
    ROOT = (Path.cwd() / "../movielens-reccomender").resolve()
elif (Path.cwd() / "configs/experiment.json").exists():
    ROOT = Path.cwd()
else:
    ROOT = (Path.cwd() / "ml-1m/movielens-reccomender").resolve()

config, model_path = select_best_pipeline(ROOT)
print("Выбрана модель по январской validation:", model_path.name)
ratings, movies, users = load_movielens(ROOT / config["data_dir"])
SEED = config["seed"]

Выбрана модель по январской validation: catboost_base.cbm


In [2]:
from movielens_recsys.retrieval import (
    fit_retrieval,
    recommend_als,
    recommend_item_knn,
    recommend_popular,
)

In [3]:
# Путь уже выбран по validation вместе с конфигом retrieval.
print("Модель для февральского теста:", model_path.name)

Модель для февральского теста: catboost_base.cbm


In [4]:
loaded_ranker = CatBoostRanker()
loaded_ranker.load_model(model_path)
BASE_FEATURES = list(config["ranking"]["features"])

print("Модель успешно загружена")

Модель успешно загружена


## История и полный ground truth
Положительные ответы не ограничиваются кандидатами retrieval.

In [5]:
february_start = pd.Timestamp("2001-02-01", tz="UTC")
february_end = pd.Timestamp("2001-03-01", tz="UTC")

february_history, february_target = split_snapshot(
    ratings,
    february_start,
    february_end,
    
)
february_ground_truth = (
    february_target[
        february_target["rating"] >= config["positive_threshold"]
    ]
    .groupby("user_id")["movie_id"]
    .apply(set)
    .to_dict()
)
february_users = sorted(
    set(february_ground_truth)
    & set(february_history["user_id"])
)
february_catalog = set(february_history["movie_id"])

## Февральский снимок
Кандидаты и признаки строятся из прошлого; target используется только для оценки.

In [6]:
february_frame = build_snapshot(
    ratings=ratings,
    users=users,
    start=february_start,
    end=february_end,
    config=config,
    seed=SEED,
)
print("February:", february_frame.shape)
display(february_frame.head())

February: (86750, 22)


,snapshot_time,user_id,movie_id,als_score,knn_score,als_rank,knn_rank,popular_rank,from_als,from_knn,...,knn_score_missing,user_rating_count,user_mean_rating,item_rating_count,item_mean_rating,gender,age,occupation,target,group_id
0,2001-02-01 00:00:00+00:00,19,3578,0.753885,7.018239,1,24,13,1,1,...,0,216,3.689815,1698,4.162544,M,1,10,0,2001-02-01_user_19
1,2001-02-01 00:00:00+00:00,19,2396,0.647479,9.881832,2,17,5,1,1,...,0,216,3.689815,2272,4.128961,M,1,10,0,2001-02-01_user_19
2,2001-02-01 00:00:00+00:00,19,223,0.555357,6.099715,3,33,38,1,1,...,0,216,3.689815,1317,3.959757,M,1,10,1,2001-02-01_user_19
3,2001-02-01 00:00:00+00:00,19,480,0.554011,18.342026,4,2,9,1,1,...,0,216,3.689815,2581,3.765982,M,1,10,1,2001-02-01_user_19
4,2001-02-01 00:00:00+00:00,19,1954,0.531674,7.242141,5,22,83,1,1,...,0,216,3.689815,1030,3.937864,M,1,10,0,2001-02-01_user_19


## Ранжирование CatBoost
В predict передаются только 11 признаков, без target. Исходный порядок Union сохраняется в отдельной таблице.

In [7]:
scored_frame = february_frame.copy()

scored_frame["catboost_score"] = loaded_ranker.predict(
    scored_frame[BASE_FEATURES]
)

scored_frame = scored_frame.sort_values(
    ["user_id", "catboost_score", "movie_id"],
    ascending=[True, False, True],
)

assert "target" not in BASE_FEATURES
assert loaded_ranker.feature_names_ == BASE_FEATURES
assert set(february_frame["user_id"]) == set(february_users)
assert february_frame.groupby("user_id").size().eq(250).all()

# Рекомендации собираем из отсортированной таблицы CatBoost.
ranked_recommendations = scored_frame.groupby("user_id")["movie_id"].apply(list).to_dict()

## Retrieval 

In [8]:
test_retrieval = fit_retrieval(
    february_history,
    positive_threshold=config["positive_threshold"],
    seed=SEED,
    n_neighbors=config["retrieval"]["item_knn_neighbors"],
    als_parameters=config["retrieval"]["als_parameters"],
)

In [9]:
popular_recommendations = {}
als_recommendations = {}
knn_recommendations = {}
original_union_recommendations = {}

for user_id in february_users:
    popular = recommend_popular(user_id, config["retrieval"]["max_candidates"], test_retrieval)
    als = recommend_als(user_id, config["retrieval"]["max_candidates"], test_retrieval)
    knn = recommend_item_knn(user_id, config["retrieval"]["max_candidates"], test_retrieval)
    popular_recommendations[user_id] = popular
    als_recommendations[user_id] = als
    knn_recommendations[user_id] = knn

    candidates = []
    for movie_id in als[:config["retrieval"]["als_candidates"]]:
        if movie_id not in candidates:
            candidates.append(movie_id)
    for movie_id in knn[:config["retrieval"]["item_knn_candidates"]]:
        if movie_id not in candidates:
            candidates.append(movie_id)
    for movie_id in popular:
        if len(candidates) >= config["retrieval"]["max_candidates"]:
            break
        if movie_id not in candidates:
            candidates.append(movie_id)
    original_union_recommendations[user_id] = candidates
    assert set(candidates) == set(ranked_recommendations[user_id])

# Состав кандидатов одинаковый; порядок после CatBoost может отличаться.
changed_users = sum(
    ranked_recommendations[u][:10] != original_union_recommendations[u][:10]
    for u in february_users
)
print("Изменился top-10 у пользователей:", changed_users, "из", len(february_users))


Изменился top-10 у пользователей: 347 из 347


## Итоговые метрики
Все модели сравниваются при одинаковых K. Таблица сохраняется для экрана результатов Streamlit.

In [10]:
test_results = []
models = {
    "MostPopular": popular_recommendations,
    "ALS": als_recommendations,
    "ItemKNN": knn_recommendations,
    "Union_ALS150_KNN100+Popular": original_union_recommendations,
    "CatBoost": ranked_recommendations,
}
for k in [5, 10, 20, 50, 100, 250]:
    for model_name, recommendations in models.items():
        _, result = evaluate_recommendations(
            recommendations=recommendations,
            ground_truth=february_ground_truth,
            eligible_users=february_users,
            catalog=february_catalog,
            model_name=model_name,
            k=k,
        )
        test_results.append(result)

test_metrics = pd.DataFrame(test_results)
display(test_metrics)

# Сохраняем итоговую таблицу без изменения параметров моделей.
report_dir = ROOT / "reports" / "ranking"
report_dir.mkdir(parents=True, exist_ok=True)
test_metrics.to_csv(report_dir / "february_test_metrics.csv", index=False)


,model,k,precision,recall,hitrate,ndcg,catalog_coverage,mean_list_length,users
0,MostPopular,5,0.077810,0.042032,0.244957,0.093503,0.025523,5.0,347
1,ALS,5,0.077810,0.050509,0.250720,0.096414,0.100462,5.0,347
2,ItemKNN,5,0.082997,0.053813,0.262248,0.101381,0.035840,5.0,347
3,Union_ALS150_KNN100+Popular,5,0.077810,0.050509,0.250720,0.096414,0.100462,5.0,347
4,CatBoost,5,0.087032,0.057497,0.282421,0.104920,0.034211,5.0,347
5,MostPopular,10,0.067723,0.063546,0.317003,0.091353,0.038012,10.0,347
6,ALS,10,0.068300,0.078552,0.337176,0.098805,0.134401,10.0,347
7,ItemKNN,10,0.068876,0.075558,0.334294,0.100076,0.055661,10.0,347
8,Union_ALS150_KNN100+Popular,10,0.068300,0.078552,0.337176,0.098805,0.134401,10.0,347
9,CatBoost,10,0.071758,0.078266,0.354467,0.102325,0.060548,10.0,347
